In [4]:
from google.colab import drive
drive.mount('/content/drive')

In [6]:
import os
import joblib
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from scipy.stats import spearmanr

DRIVE_DIR = '/content/drive/MyDrive/kb_challenge'
SAVE_DIR = os.path.join('/content/drive/MyDrive/data', 'saved_models_v6')

final_model = joblib.load(os.path.join(SAVE_DIR, 'final_model_for_shap.pkl'))
final_calibrator = joblib.load(os.path.join(SAVE_DIR, 'final_calibrator_for_service.pkl'))
FEATURE_COLS = joblib.load(os.path.join(SAVE_DIR, 'feature_cols.pkl'))

holdout = pd.read_parquet(os.path.join(DRIVE_DIR, '2단계_2025홀드아웃_피처완료.parquet'))

# 1단계_위험점수 정상적으로 채워졌는지 먼저 확인 (예전엔 100% 결측이었던 부분)
print('1단계_위험점수 결측률:', holdout['1단계_위험점수'].isna().mean())

X_2025 = holdout[FEATURE_COLS]
y_2025 = holdout['최종_보정라벨'].values

pred_proba = final_calibrator.predict_proba(X_2025)[:, 1]

print('=== 2025 진짜 미래데이터 검증 ===')
print('ROC-AUC:', roc_auc_score(y_2025, pred_proba))
print('PR-AUC:', average_precision_score(y_2025, pred_proba))
print('Spearman:', spearmanr(pred_proba, y_2025)[0])
print('Brier:', brier_score_loss(y_2025, pred_proba))

# 분기별로도 쪼개서 보면 좋음
holdout['pred_proba'] = pred_proba
for q in sorted(holdout['기준_년분기'].unique()):
    sub = holdout[holdout['기준_년분기'] == q]
    print(f"{q}: AUC={roc_auc_score(sub['최종_보정라벨'], sub['pred_proba']):.4f}")

1단계_위험점수 결측률: 0.33512246022824854
=== 2025 진짜 미래데이터 검증 ===
ROC-AUC: 0.6066671376348337
PR-AUC: 0.04418005745044405
Spearman: 0.06507230678829959
Brier: 0.031566082742714975
2025Q1: AUC=0.6111
2025Q2: AUC=0.6389
2025Q3: AUC=0.6288


In [7]:
import pandas as pd

# ===== 1. 전체 요약 지표 =====
overall_metrics = pd.DataFrame([{
    '구분': '전체(2025Q1~Q3)',
    'ROC_AUC': roc_auc_score(y_2025, pred_proba),
    'PR_AUC': average_precision_score(y_2025, pred_proba),
    'Spearman': spearmanr(pred_proba, y_2025)[0],
    'Brier': brier_score_loss(y_2025, pred_proba),
    '1단계_위험점수_결측률': holdout['1단계_위험점수'].isna().mean(),
    'n_행수': len(holdout),
}])

# ===== 2. 분기별 지표 =====
quarterly_rows = []
for q in sorted(holdout['기준_년분기'].unique()):
    sub = holdout[holdout['기준_년분기'] == q]
    quarterly_rows.append({
        '구분': q,
        'ROC_AUC': roc_auc_score(sub['최종_보정라벨'], sub['pred_proba']),
        'PR_AUC': average_precision_score(sub['최종_보정라벨'], sub['pred_proba']),
        'Spearman': spearmanr(sub['pred_proba'], sub['최종_보정라벨'])[0],
        'Brier': brier_score_loss(sub['최종_보정라벨'], sub['pred_proba']),
        '1단계_위험점수_결측률': sub['1단계_위험점수'].isna().mean(),
        'n_행수': len(sub),
    })
quarterly_metrics = pd.DataFrame(quarterly_rows)

# ===== 3. 합쳐서 저장 =====
final_result = pd.concat([overall_metrics, quarterly_metrics], ignore_index=True)
print(final_result)

import os
OUTPUT_PATH = os.path.join('/content/drive/MyDrive/data', '2025_최종검증_결과.csv')
final_result.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'저장 완료: {OUTPUT_PATH}')

              구분   ROC_AUC    PR_AUC  Spearman     Brier  1단계_위험점수_결측률  \
0  전체(2025Q1~Q3)  0.606667  0.044180  0.065072  0.031566      0.335122   
1         2025Q1  0.611055  0.052809  0.070892  0.033693      1.000000   
2         2025Q2  0.638916  0.055520  0.090198  0.034913      0.000000   
3         2025Q3  0.628796  0.037393  0.068987  0.026076      0.000000   

      n_행수  
0  1612891  
1   540516  
2   536114  
3   536261  
저장 완료: /content/drive/MyDrive/data/2025_최종검증_결과.csv
